### Exercise 1: The Functional API and Feature Extraction
The Functional API is ideal for models with non-linear topologies, multiple inputs, or multiple outputs.
*   **Part A:** Build a multi-input, multi-output model. Create a model that takes two inputs (e.g., a `text_input` of shape `(1000,)` and an `image_input` of shape `(28, 28, 1)`). Pass them through a dense layer and a convolutional layer respectively, flatten the image output, concatenate the two resulting tensors, and output two separate predictions (e.g., a binary `priority` score and a 5-class `category` classification).
*   **Part B:** Practice **feature extraction**. Using the model you just built, write code to instantiate a *new* model that takes the same inputs but outputs the intermediate concatenated tensor instead of the final predictions. 
*   **Hint:** You will need to use `model.layers` to find your concatenation layer, and then use its `.output` attribute to define your new `keras.Model(inputs=..., outputs=...)`.

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "jax"

import keras

text_input = keras.Input(shape=(1000,), name="text_input_layer")
image_input = keras.Input(
    shape=(
        28,
        28,
        1,
    ),
    name="image_input",
)

dense_layer = keras.layers.Dense(64, activation="relu", name="dense_layer_1")(
    text_input
)
convolutional_layer = keras.layers.Convolution2D(
    filters=64,
    kernel_size=(2, 2),
    strides=(1, 1),
    activation="relu",
    name="convolutional_layer",
)(image_input)
flatten_image = keras.layers.Flatten()(convolutional_layer)

concat_both_inputs = keras.layers.Concatenate()([flatten_image, dense_layer])

priority_layer = keras.layers.Dense(1, activation="sigmoid", name="priority")(
    concat_both_inputs
)
category_layer = keras.layers.Dense(5, activation="softmax", name="category")(
    concat_both_inputs
)

multi_input_multi_output_model = keras.Model(
    inputs=[text_input, image_input], outputs=[priority_layer, category_layer]
)

multi_input_multi_output_model.compile(
    optimizer=keras.optimizers.Adam(),
    loss={
        "priority": "binary_crossentropy",
        "category": "categorical_crossentropy",
    },
    metrics={
        "priority": ["accuracy"],
        "category": ["accuracy"],
    },
)

multi_input_multi_output_model.summary()

In [ ]:
import numpy as np
from keras.utils import to_categorical

num_samples = 100

dummy_text_data = np.random.random(size=(num_samples, 1000))

dummy_image_data = np.random.random(size=(num_samples, 28, 28, 1))

dummy_priority_targets = np.random.randint(2, size=(num_samples, 1))

dummy_category_labels = np.random.randint(5, size=(num_samples, 1))

dummy_category_targets = to_categorical(dummy_category_labels, num_classes=5)

In [ ]:
history = multi_input_multi_output_model.fit(
    x={"text_input_layer": dummy_text_data, "image_input": dummy_image_data},
    y={"priority": dummy_priority_targets, "category": dummy_category_targets},
    epochs=100,
    batch_size=32,
)

In [ ]:
import matplotlib.pyplot as plt

# 2. Extract the loss and accuracy metrics from the history dictionary
history_dict = history.history
epochs = range(1, len(history_dict["loss"]) + 1)
# --- Plot the Losses ---
plt.figure(figsize=(12, 5))
# Plot total loss, priority loss, and category loss
plt.subplot(1, 2, 1)  # 1 row, 2 columns, 1st plot
plt.plot(epochs, history_dict["loss"], "b-", label="Total Loss")
plt.plot(epochs, history_dict["priority_loss"], "r--", label="Priority Loss")
plt.plot(epochs, history_dict["category_loss"], "g--", label="Category Loss")
plt.title("Training Losses")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
# --- Plot the Accuracies ---
plt.subplot(1, 2, 2)  # 1 row, 2 columns, 2nd plot
# Plot priority accuracy and category accuracy
plt.plot(epochs, history_dict["priority_accuracy"], "r-", label="Priority Accuracy")
plt.plot(epochs, history_dict["category_accuracy"], "g-", label="Category Accuracy")
plt.title("Training Accuracies")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
# Display the plots
plt.tight_layout()
plt.show()


### Exercise 2: Model Subclassing
Model subclassing gives you the ultimate flexibility to write everything yourself from scratch.
*   **Task:** Rewrite the multi-output model you built in Exercise 1, but this time do it by subclassing `keras.Model`. 
*   **Hint:** You need to define your layers in the `__init__()` method and implement the forward pass logic in the `call()` method. Remember that your `call()` method should accept a dictionary or list of inputs and return your multiple outputs.

In [ ]:
class ModelSubclassing(keras.Model):
    def __init__(self):
        super().__init__()
        self.dense_layer = keras.layers.Dense(
            64, activation="relu", name="dense_layer_1"
        )
        self.convolutional_layer = keras.layers.Convolution2D(
            filters=64,
            kernel_size=(2, 2),
            strides=(1, 1),
            activation="relu",
            name="convolutional_layer",
        )
        self.flatten_image = keras.layers.Flatten()
        self.concat_both_inputs = keras.layers.Concatenate()
        self.priority_layer = keras.layers.Dense(
            1, activation="sigmoid", name="priority"
        )
        self.category_layer = keras.layers.Dense(
            5, activation="softmax", name="category"
        )

    def call(self, inputs):
        text_input = inputs["text_input_layer"]
        image_input = inputs["image_input"]
        dense_output = self.dense_layer(text_input)
        convolutional_output = self.convolutional_layer(image_input)
        flatten_image_output = self.flatten_image(convolutional_output)
        concat_output = self.concat_both_inputs([flatten_image_output, dense_output])
        priority_output = self.priority_layer(concat_output)
        category_output = self.category_layer(concat_output)
        return {"priority": priority_output, "category": category_output}


model_subclassing = ModelSubclassing()
model_subclassing.compile(
    optimizer=keras.optimizers.Adam(),
    loss={
        "priority": "binary_crossentropy",
        "category": "categorical_crossentropy",
    },
    metrics={
        "priority": ["accuracy"],
        "category": ["accuracy"],
    },
)

In [ ]:
import numpy as np
from keras.utils import to_categorical

text_data = np.random.random(size=(num_samples, 1000))
image_data = np.random.random(size=(num_samples, 28, 28, 1))
priority_labels = np.random.randint(2, size=(num_samples, 1))
category_labels = np.random.randint(5, size=(num_samples, 1))
category_labels = to_categorical(category_labels, num_classes=5)

history_subclassing = model_subclassing.fit(
    x={"text_input_layer": text_data, "image_input": image_data},
    y={"priority": priority_labels, "category": category_labels},
    epochs=10,
    batch_size=32,
)

In [ ]:
# Plot the training history for the subclassing model
history_subclassing_dict = history_subclassing.history
epochs_subclassing = range(1, len(history_subclassing_dict["loss"]) + 1)
plt.figure(figsize=(12, 5))
# Plot total loss, priority loss, and category loss
plt.subplot(1, 2, 1)  # 1 row, 2 columns
plt.plot(epochs_subclassing, history_subclassing_dict["loss"], "b-", label="Total Loss")
plt.plot(
    epochs_subclassing,``
    history_subclassing_dict["priority_loss"],
    "r--",
    label="Priority Loss",
)
plt.plot(
    epochs_subclassing,
    history_subclassing_dict["category_loss"],
    "g--",
    label="Category Loss",
)
plt.title("Training Losses (Subclassing Model)")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
# Plot priority accuracy and category accuracy
plt.subplot(1, 2, 2)  # 1 row, 2 columns
plt.plot(
    epochs_subclassing,
    history_subclassing_dict["priority_accuracy"],
    "r-",
    label="Priority Accuracy",
)
plt.plot(
    epochs_subclassing,
    history_subclassing_dict["category_accuracy"],
    "g-",
    label="Category Accuracy",
)
plt.title("Training Accuracies (Subclassing Model)")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


### Exercise 3: Writing a Custom Metric
When built-in metrics (like accuracy) don't capture what you need to measure, you can write your own by subclassing `keras.metrics.Metric`.
*   **Task:** Implement a custom metric class called `MeanAbsoluteErrorTracker`. 
*   **Hint:** You must implement four specific methods: 
    *   `__init__` to create state variables using `self.add_weight()`.
    *   `update_state(y_true, y_pred)` to define the logic for updating the state variables per batch.
    *   `result()` to return the current metric value.
    *   `reset_state()` to clear the metric between epochs.

In [ ]:
import keras
import tensorflow as tf
import numpy as np
from keras import ops

num_samples = 100
text_data = np.random.random(size=(num_samples, 1000))
priority_labels = np.random.randint(2, size=(num_samples, 1)).astype(np.float32)


class MeanAbsoluteErrorTracker(keras.metrics.Metric):
    def __init__(self, name="mean_absolute_error_tracker", **kwargs):
        super().__init__(name=name, **kwargs)
        self.total_absolute_error = self.add_weight(
            name="total_absolute_error", initializer="zeros"
        )
        self.count = self.add_weight(name="count", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        logging.info("y_true shape: %s", y_true.shape)
        absolute_errors = ops.abs(y_true - y_pred)
        sum_absolute_errors = ops.sum(absolute_errors)
        num_samples = ops.shape(y_true)[0]
        self.total_absolute_error.assign_add(sum_absolute_errors)
        self.count.assign_add(ops.cast(num_samples, dtype=self.count.dtype))

    # Calculate the mean absolute error
    def result(self):
        return self.total_absolute_error / self.count

    def reset_states(self):
        self.total_absolute_error.assign(0.0)
        self.count.assign(0.0)


mean_absolute_error_tracker = MeanAbsoluteErrorTracker()

model = keras.Sequential(
    [
        keras.layers.Input(shape=(1000,), name="input_layer"),
        keras.layers.Dense(64, activation="relu", name="dense_layer_1"),
        keras.layers.Dense(1, activation="sigmoid", name="output_layer"),
    ]
)

model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="binary_crossentropy",
    metrics=[mean_absolute_error_tracker],
)
model.fit(text_data, priority_labels, epochs=10, batch_size=32)


### Exercise 4: Building a Smart Callback
Callbacks allow you to self-introspect and dynamically take action during the `fit()` loop.
*   **Task:** Write a custom callback by subclassing `keras.callbacks.Callback` that tracks the highest loss seen across all batches within a single epoch. Have it print a warning message at the end of the epoch if the maximum batch loss exceeded a certain threshold (e.g., `1.5`).
*   **Hint:** You will need to implement the `on_epoch_begin()` method to reset your maximum loss tracker, the `on_batch_end(batch, logs)` method to check the current batch's loss, and the `on_epoch_end(epoch, logs)` method to print your warning.

In [ ]:
import logging
import keras

logging.basicConfig(level=logging.INFO)


class SmartCallback(keras.callbacks.Callback):
    def __init__(self, threshold=0.5):
        super().__init__()
        self._threshold = threshold
        self._epoch = 0
        self._best_loss = float("inf")

    def on_epoch_end(self, epoch, logs=None):
        logging.info(
            f"Epoch {epoch+1} ended. Current loss: {logs.get('loss'):.4f}. Best loss so far: {self._best_loss:.4f}."
        )
        if self._best_loss < self._threshold:
            logging.warning(
                f"Best loss {self._best_loss:.4f} is below the threshold {self._threshold}. Consider stopping training."
            )

    def on_epoch_begin(self, epoch, logs=None):
        self._best_loss = float("inf")
        self._epoch = epoch

    def on_batch_end(self, batch, logs=None):
        if logs is not None:
            current_loss = logs.get("loss")
            if not current_loss:
                return
            if current_loss < self._best_loss:
                self._best_loss = current_loss


smart_callback = SmartCallback(threshold=0.5)
model.fit(
    text_data, priority_labels, epochs=10, batch_size=32, callbacks=[smart_callback]
)

### Exercise 5: Customizing the `fit()` method
Overriding the `train_step()` method allows you to inject custom training algorithms while still taking advantage of the convenient `model.fit()` API (like progress bars and callbacks).
*   **Task:** Subclass `keras.Model` and override the `train_step(self, data)` method. Implement the forward pass, loss calculation, gradient retrieval, and optimizer application. 
*   **Hint:** While Keras layers are backend-agnostic, gradient retrieval is backend-specific. Depending on your configured backend, you will need to use `tf.GradientTape()` for TensorFlow, `loss.backward()` for PyTorch, or `jax.value_and_grad()` for JAX. Don't forget to use `self.trainable_weights` when updating the model's parameters.

In [ ]:
import os
import jax

# Set the environment variable to use the JAX backend for Keras
os.environ["KERAS_BACKEND"] = "jax"


class CustomFitModel(keras.Model):
    def __init__(self):
        super().__init__()
        self.dense_layer = keras.layers.Dense(
            64, activation="relu", name="dense_layer_1"
        )
        self.output_layer = keras.layers.Dense(
            1, activation="sigmoid", name="output_layer"
        )

    def compute_loss_and_updates(
        self,
        trainable_weights,
        non_trainable_variables,
        inputs,
        targets,
        training=False,
    ):
        # Stateless forward pass
        predictions, non_trainable_variables = self.stateless_call(
            trainable_variables=trainable_weights,
            non_trainable_variables=non_trainable_variables,
            inputs=inputs,
            training=training,
        )

        # Compute loss using JAX operations
        loss = self.compute_loss(predictions, targets)
        # Return loss and any auxiliary outputs (like predictions and updated non-trainable vars)
        return loss, predictions, non_trainable_variables

    def call(self, inputs):
        x = self.dense_layer(inputs)
        return self.output_layer(x)

    def train_step(self, data):
        x, y = data
        pass

### Datasets for Practice
Working with larger, non-toy datasets will make your custom training loops and model architectures much more realistic. Here are three datasets from the book that you can download directly:

**1. The Dogs vs. Cats Dataset (Image Classification)**
This is a classic computer vision dataset containing 25,000 images of dogs and cats. It is great for practicing data loading and building custom architectures.
*   **How to download:** You can download it using the Kaggle API (pre-installed in Colab as `kagglehub`):
    ```python
    import kagglehub
    kagglehub.login() # Requires a Kaggle account/API key
    download_path = kagglehub.competition_download("dogs-vs-cats")
    ```

**2. The Jena Climate Dataset (Timeseries Forecasting)**
This dataset consists of 14 different weather quantities recorded every 10 minutes over several years. It is highly suited for practicing low-level custom training loops because sequence data requires careful batching.
*   **How to download:**
    ```bash
    !wget https://s3.amazonaws.com/keras-datasets/jena_climate_2009_2016.csv.zip
    !unzip jena_climate_2009_2016.csv.zip
    ```

**3. The Oxford-IIIT Pets Dataset (Image Segmentation & Classification)**
This dataset contains 7,390 pictures of various breeds of cats and dogs, alongside foreground-background segmentation masks. It is perfect for building complex multi-output models.
*   **How to download:**
    ```bash
    !wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
    !wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz
    !tar -xf images.tar.gz
    !tar -xf annotations.tar.gz
    ```

---

### Advanced Exercises for Chapter 7

#### Exercise 1: A Complete Low-Level Training Loop from Scratch
Instead of just overriding `train_step()` and relying on `model.fit()`, write the entire training algorithm yourself from scratch using TensorFlow.
*   **Dataset:** Jena Climate Dataset or Dogs vs. Cats.
*   **Task:** Write a Python `for` loop that iterates over a specified number of epochs, and an inner `for` loop that iterates over batches of data yielded by a `tf.data.Dataset`. 
*   **Requirements:** Inside the inner loop, open a `tf.GradientTape()` scope, run the forward pass, compute the loss, retrieve the gradients using `tape.gradient()`, and apply them to the model's weights using an optimizer (e.g., `optimizer.apply_gradients()`). 
*   **Bonus:** Add a custom training metric that is updated manually after each batch using `metric.update_state()` and reset at the end of each epoch using `metric.reset_state()`.

In [5]:
import kagglehub
import os

os.environ["KAGGLE_API_TOKEN"] = "KGAT_b16fca105497fc0a804ee5d19d4699e6"

kagglehub.login()  # Requires a Kaggle account/API key

# Download latest version
path = kagglehub.dataset_download("shaunthesheep/microsoft-catsvsdogs-dataset")

print("Path to dataset files:", path)

100%|██████████| 788M/788M [00:22<00:00, 36.1MB/s] 

Extracting files...


Path to dataset files: /Users/lucaswerner/.cache/kagglehub/datasets/shaunthesheep/microsoft-catsvsdogs-dataset/versions/1


In [3]:
# Explore the downloaded dataset (e.g., list files, load data, etc.)
import logging
import zipfile

logging.basicConfig(level=logging.INFO)

for root, dirs, files in os.walk(download_path):
    for filename in files:
        if filename.endswith(".zip"):
          # Unzip the dataset and log its contents
          with zipfile.ZipFile(os.path.join(root, filename), "r") as zip_ref:
            # Move the unzipped files to a specific directory if needed
            zip_ref.extractall(os.path.join(root, download_path))

In [7]:
# Move the content of the dataset into a local folder so I don't need to download it every time.
dataset_target_dir = os.path.join(os.path.dirname(os.getcwd()),'datasets','dogs-vs-cats')
dataset_source_dir = path
# Ensure target directory exists (optional)
os.makedirs(dataset_target_dir, exist_ok=True)
# Iterate over all items in the source directory
for item_name in os.listdir(download_path):
    source_item = os.path.join(download_path, item_name)
    target_item = os.path.join(target_dir, item_name)
    
    # Copy each item (file or sub-directory) to the target directory
    shutil.move(source_item, target_item)

In [ ]:
import os
import shutil

# The test directory doesn't contain any information about the labels, therefore to train the model we need to test it with the train directory.
train_dir = os.path.join(download_path, 'train')

# Define paths for train subdirectories
train_cat_dir = os.path.join(train_dir, 'cat')
train_dog_dir = os.path.join(train_dir, 'dog')

# Create subdirectories for train if they don't exist
os.makedirs(train_cat_dir, exist_ok=True)
os.makedirs(train_dog_dir, exist_ok=True)

# Move cat images to cat subdirectory
for filename in os.listdir(train_dir):
	if filename.startswith('cat.'):
		shutil.move(os.path.join(train_dir, filename), os.path.join(train_cat_dir, filename))
	elif filename.startswith('dog.'):
		shutil.move(os.path.join(train_dir, filename), os.path.join(train_dog_dir, filename))

In [ ]:
import os
from PIL import Image

import matplotlib.pyplot as plt

# Define the path to the train images
train_dir = os.path.join(download_path, 'train')

# Get a list of image files (e.g., first 4 dog and cat images)
dog_images = [f for f in os.listdir(train_dir) if f.startswith('dog')][:2]
cat_images = [f for f in os.listdir(train_dir) if f.startswith('cat')][:2]

# Display the images
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for i, img_file in enumerate(dog_images + cat_images):
	img_path = os.path.join(train_dir, img_file)
	img = Image.open(img_path)
	ax = axes[i // 2, i % 2]
	ax.imshow(img)
	ax.set_title(img_file)
	ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf

# Define image size and batch size
img_size = (128, 128)
batch_size = 32

# Load images from train directory using image_dataset_from_directory
train_dataset = tf.keras.utils.image_dataset_from_directory(
	train_dir,
	seed=42,
	image_size=img_size,
	batch_size=batch_size,
	label_mode='binary',  # Binary classification: dog vs cat
	subset='training',
  class_names=['cat', 'dog'],
	validation_split=0.2
)

# Load test images
test_dir = os.path.join(data_directory, 'test1')
test_dataset = tf.keras.utils.image_dataset_from_directory(
	test_dir,
	seed=42,
	image_size=img_size,
	batch_size=batch_size,
	label_mode=None,  # No labels for test set
	shuffle=False
)

# Normalize pixel values to [0, 1]
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset = test_dataset.map(lambda x: normalization_layer(x))

# Prefetch datasets for performance
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.prefetch(tf.data.AUTOTUNE)

In [ ]:
# Load and preprocess the images using tensorflow datasets (e.g., resize, normalize, etc.)
from tensorflow_data import Dataset
from tensorflow_data import image_dataset_from_directory

#### Exercise 2: Hybrid Model Building (Functional API + Subclassing)
Chapter 7 notes that mixing the Functional API with subclassed layers provides "the best of both worlds: high development flexibility while retaining the advantages of the Functional API". 
*   **Dataset:** Any of the above.
*   **Task:** Subclass `keras.layers.Layer` to create a highly custom, reusable block (for example, a layer that dynamically adds random noise to its inputs only during training, or a custom residual block). 
*   **Requirements:** Then, use the **Functional API** (`keras.Input`, `keras.Model`) to build a larger model that incorporates several instances of your custom subclassed layer. 

#### Exercise 3: A Multi-Input, Multi-Output Architecture
The Functional API is necessary when models don't follow a straight line. 
*   **Dataset:** Oxford-IIIT Pets Dataset.
*   **Task:** Build a dual-headed model. The model should take an image as input and then branch into two separate output heads. 
*   **Requirements:** The first output head should be a `Dense` layer designed to predict the class of the pet (classification). The second output head should use convolutional layers to predict the segmentation mask of the pet (predicting whether each pixel is foreground, background, or contour). Compile the model with two different loss functions and two different metrics, assigning them to the specific output layers via dictionaries in `model.compile()`.

#### Exercise 4: Advanced Callbacks and TensorBoard Integration
Introspection during training is crucial for advanced workflows.
*   **Dataset:** Dogs vs. Cats.
*   **Task:** Write a highly custom callback by subclassing `keras.callbacks.Callback` that interfaces with TensorBoard.
*   **Requirements:** Set up the callback to evaluate a small, fixed validation batch at the end of every epoch. Have your callback use `tf.summary` to write the model's actual image predictions (e.g., the image alongside its predicted label) directly to TensorBoard so you can visually monitor how the model's understanding improves over time. Ensure you integrate this custom callback into the standard `model.fit()` loop.